# Preprocessing

The text content is not in its correct format - it is currently in a word format. This data cannot be trained on, so we must put it through a process called tokenisation to convert it into a format the model will understand (this will be explained throughout the notebooks). 

Of course, tokenisation is not possible without first cleaning up the data. 

## Imports

Firstly, let's import our data as a CSV. 

In [13]:
import pandas as pd
df: pd.DataFrame = pd.read_csv("csv/all_data.csv")

In this section, I am using a library called NLTK (Natural Language Toolkit), which contains utilities for being able to convert sentences into tokens.   

In [14]:
import nltk
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
import re
import contractions

This downloads samples and datasets for the nltk to use. I'm not particularly sure about what it does, NLTK functions throw an error asking for this to be downloaded. 

In [15]:
nltk.download('stopwords')
nltk.download('twitter_samples')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package twitter_samples to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Stopwords

Stopwords are words that carry semantic meaning and can make analysis of the words hard. It includes such articles ("a", "the", ...), conjunctions ("and", "or", ...), prepositions ("in", "on", ...) and other such words. 

By removing stopwords, we are able to remove any tokens/words that do not hold minimal meaning. 

In [16]:
stop_words = set(stopwords.words('english'))

There are some words that can drastically change the meaning if removed. 

For example:

"That is not good" (negative) ----removing stopword---> "That is good" (positive) *[not what we want]*

By excluding the negations from the stopwords, we will be able to keep the sentiment of the text block

In [17]:
negations = {'no', 'not', 'nor', 'neither', 'never', 'none',
            "don't", "won't", "can't", "isn't", "aren't", "wasn't"}
stop_words = stop_words - negations

## Tokenising

By tokenising, we are able to convert a long block of text into individual words. 

In [18]:
tokeniser = TweetTokenizer(strip_handles=True, reduce_len=True)

In [19]:
def clean_text_content(text):
    if pd.isna(text):
        return ""
    text = str(text)

    text = re.sub(r'[^\x00-\x7F]+', '', text)     # remove non-ASCII
    text = re.sub(r'http\S+|www\.\S+', '', text)  # remove URLs
    text = re.sub(r'#\w+', '', text)              # remove hashtags
    text = re.sub(r'&\w+;', '', text)             # remove HTML entities
    text = contractions.fix(text)                 # expand contractions
    text = text.lower().strip()
    return text

df['Mutated Text Content'] = df['Text Content'].apply(clean_text_content)
df.head()

,Unnamed: 0,Sentiment,Text Content,tweet_length,Mutated Text Content
0,0,Negative,Fuck Overwatch,14,fuck overwatch
1,1,Irrelevant,Have you ever been in contact with someone and...,119,have you ever been in contact with someone and...
2,2,Positive,I love @ Rainbow6Game and I'm a.. I'm.. I real...,189,i love @ rainbow6game and i am a.. i am.. i re...
3,3,Negative,"Um. An Australia that has been ""reshaped"" by A...",135,"um. an australia that has been ""reshaped"" by a..."
4,4,Irrelevant,It is not the first time that the EU Commissio...,70,it is not the first time that the eu commissio...


In [20]:
def tokenise_text_content(text):
    if not text:
        return []
    tokens = tokeniser.tokenize(text)
    tokens = [re.sub(r'[^\w\s]', '', w) for w in tokens]  # remove punctuation
    tokens = [w for w in tokens if w and w not in stop_words]
    return tokens

df["Text Tokens"] = df["Mutated Text Content"].apply(tokenise_text_content)

# remove rows with no usable tokens like empty
df = df[df["Text Tokens"].apply(len) > 0].reset_index(drop=True)

df.head()

,Unnamed: 0,Sentiment,Text Content,tweet_length,Mutated Text Content,Text Tokens
0,0,Negative,Fuck Overwatch,14,fuck overwatch,"[fuck, overwatch]"
1,1,Irrelevant,Have you ever been in contact with someone and...,119,have you ever been in contact with someone and...,"[ever, contact, someone, phone, goes, stop, re..."
2,2,Positive,I love @ Rainbow6Game and I'm a.. I'm.. I real...,189,i love @ rainbow6game and i am a.. i am.. i re...,"[love, rainbow, 6game, really, love, want, get..."
3,3,Negative,"Um. An Australia that has been ""reshaped"" by A...",135,"um. an australia that has been ""reshaped"" by a...","[um, australia, reshaped, afr, microsoft, one,..."
4,4,Irrelevant,It is not the first time that the EU Commissio...,70,it is not the first time that the eu commissio...,"[not, first, time, eu, commission, taken, step]"


## Classification of sentiment

As previously mentioned, there are ~~4~~ 3 primary types of sentiment classification: 
- Positive
- Neutral/Irrelevant
- Negative

Under initial testing, neutral/irrelevant data is not accurate to be used, as it skews the dataset (in terms of size), but also it is not a great method of determining sentiment. 

Within the set, we can use binary classification as so:
- Positive (1)
- Negative (0) (it only made sense if neutral was 0, not Negative)

Firstly, lets remove the sentiments that are not needed (Neutral/Irrelevant). 

In [21]:
df = df[~df['Sentiment'].isin(['Neutral', 'Irrelevant'])]

Now, we need to map the string content into a numerical form, something that the models can understand. 

In [22]:
s_map = {
    'Positive': 1,
    'Negative': 0,
}

df['Sentiment'] = df['Sentiment'].map(s_map)
df.head()

,Unnamed: 0,Sentiment,Text Content,tweet_length,Mutated Text Content,Text Tokens
0,0,0,Fuck Overwatch,14,fuck overwatch,"[fuck, overwatch]"
2,2,1,I love @ Rainbow6Game and I'm a.. I'm.. I real...,189,i love @ rainbow6game and i am a.. i am.. i re...,"[love, rainbow, 6game, really, love, want, get..."
3,3,0,"Um. An Australia that has been ""reshaped"" by A...",135,"um. an australia that has been ""reshaped"" by a...","[um, australia, reshaped, afr, microsoft, one,..."
6,6,0,The RhandlerR RhandlerR Hey... could you expla...,103,the rhandlerr rhandlerr hey... could you expla...,"[rhandlerr, rhandlerr, hey, could, explain, he..."
9,9,0,@GhostRecon there's unbearable lag on UI and N...,81,@ghostrecon there is unbearable lag on ui and ...,"[unbearable, lag, ui, npc, cutscenes, yikes]"


# Finish

Preprocessing is completed. Let's save into a .csv to be modified in feature_engineering

In [23]:
df.to_csv('csv/preprocess.csv', index=False)